# 8장 실습 ② — Conv2D에 활성화 함수를 빼면

**Keras 3 판**

옛 강의 노트북 240개 중 **22개에 있던 결함**을 재현합니다.

> 점검 기록에는 이것을 *"수학적으로 선형"*이라고 적어 두었습니다.
> **그 표현이 정확한지** 이 노트북이 확인합니다.

## 8.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 8.1 도형 데이터

28×28 회색조 영상에 원·사각형·삼각형 중 하나가 그려져 있습니다.
**위치가 매번 다릅니다.** 그것이 이 장의 핵심입니다.

In [ ]:
# 도형 데이터 — 인터넷 없이 만든다. 그리고 **위치를 마음대로 흔들 수 있다.**
# MNIST는 숫자가 대체로 가운데 있어서 CNN이 왜 필요한지가 잘 안 드러난다.
x, y = data.shapes(n=6000, seed=42, shift=6)
s = data.split(x, y, val_ratio=0.15, test_ratio=0.15, seed=42)
print(s.summary())

fig = plot.image_grid(s.x_train, s.y_train, n=24, cols=8,
                      class_names=list(data.SHAPE_CLASSES))
plt.show()

## 8.2 학습 함수 — 여기만 판마다 다릅니다

`kind` 로 DNN / CNN / 선형 모델을 만듭니다.
**PyTorch 판에서 `permute` 한 줄이 더 있는 것**에 주목하십시오 — 채널 순서가
다르기 때문입니다.

In [ ]:
import keras
from keras import layers

dlbook.set_seed(42)

def train(kind, split=None, units=(256, 128), conv_act="relu", pool="max",
          head_act="relu", n_classes=3, epochs=15, bs=64, lr=0.001, seed=42):
    """모델을 만들어 학습시키고 (시험 정확도, 파라미터 수)를 돌려준다.

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    sp = split if split is not None else s
    dlbook.set_seed(seed)
    shape = sp.x_train.shape[1:]

    if kind == "dnn":
        ls = [layers.Input(shape=shape), layers.Flatten()] + \
             [layers.Dense(u, activation="relu") for u in units]
    elif kind == "linear":
        ls = [layers.Input(shape=shape), layers.Flatten()]
    else:                                        # cnn
        Pool = layers.MaxPooling2D if pool == "max" else layers.AveragePooling2D
        ls = [layers.Input(shape=shape)]
        for f in (16, 32):
            ls.append(layers.Conv2D(f, 3, activation=conv_act, padding="same"))
            ls.append(Pool(2))
        ls += [layers.Flatten(), layers.Dense(64, activation=head_act)]
    ls.append(layers.Dense(n_classes, activation="softmax"))

    model = keras.Sequential(ls)
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(sp.x_train, sp.y_train, validation_data=(sp.x_val, sp.y_val),
              epochs=dlbook.smoke.epochs(epochs), batch_size=bs, verbose=0)
    pred = model.predict(sp.x_test, verbose=0).argmax(1)
    return metrics.accuracy(sp.y_test, pred), model.count_params()

## 8.3 활성화 함수를 빼 봅니다

`Conv2D` 는 `activation` 을 안 주면 **활성화 함수를 적용하지 않습니다.**
기본값이 `None` 입니다.

In [ ]:
# ★ 옛 노트북 240개 중 22개에 있던 결함을 재현한다.
print(f"{'Conv 활성화':<12}{'풀링':<8}{'Dense 활성화':<14}{'시험 정확도':>12}")
cases = [("relu", "max", "relu"), (None, "max", "relu"), (None, "max", None),
         (None, "avg", "relu"), (None, "avg", None)]
for ca, pl, ha in cases:
    acc, _ = train("cnn", conv_act=ca, pool=pl, head_act=ha)
    print(f"{str(ca):<12}{pl:<8}{str(ha):<14}{acc:>12.3f}")
    dlbook.record(f"ch08_noact_{ca}_{pl}_{ha}_acc", acc)

base, _ = train("linear")
dlbook.record("ch08_linear_baseline_acc", base)
print(f"{'(선형 기준선: Flatten→Dense)':<34}{base:>12.3f}")
print()
print("→ 활성화를 빼도 **최대 풀링이 비선형이라** 완전히 무너지지 않습니다.")
print("→ **평균 풀링으로 바꾸면** 전체가 선형이 되어 기준선과 같아집니다.")
print("→ 결함 목록의 '수학적으로 선형'이라는 표현은 부정확했습니다. (본문 §8.7)")

## 정리

- 활성화 함수를 빼도 **최대 풀링이 비선형이라** 완전히 무너지지 않습니다.
- **평균 풀링으로 바꾸면** 전체가 선형이 되어 **선형 기준선과 같아집니다.**
- 그러니 *"활성화 함수를 빼면 선형이 된다"* 는 말은
  **다른 비선형 연산이 하나도 없을 때만** 맞습니다.
- 그래도 이 결함은 심각합니다. **조용히 3%포인트를 잃기** 때문입니다.
  요란하게 실패하면 고치지만, 조용히 나빠지면 그대로 갑니다.

### 연습

1. 넷째 줄과 선형 기준선의 값이 정확히 같습니까. 다르다면 몇째 자리에서 다릅니까.
2. Conv 층을 4개로 늘리면 표가 어떻게 달라집니까.
3. 활성화 함수를 `sigmoid` 로 주면 어떻게 됩니까. 5장 §5.4를 떠올려 보십시오.